# TODOs
- maps split have data at lon > 200deg

To make this work for the JupyterHub server through the browser, see the follwing link:
https://docs.astral.sh/uv/guides/integration/jupyter/#using-jupyter-within-a-project

In [1]:
%load_ext autoreload
%autoreload 2

# training v1

In [ ]:
# ---
# jupyter:
#   jupytext:
#     cell_metadata_filter: -all
#     custom_cell_magics: kql
#     text_representation:
#       extension: .py
#       format_name: percent
#       format_version: '1.3'
#       jupytext_version: 1.11.2
# ---


# %%
from dataclasses import dataclass, field
from typing import Literal, Generator, Iterator

from loguru import logger
import sys
from pathlib import Path
import os
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold
import numpy as np
import catboost as cb
#import functions from preprocessing.py script
import yaml
from functools import lru_cache
import xarray as xr



# %%
# global variables
ROOT = Path.cwd().parent
DATA_PATH = ROOT / "data/training/GLODAPv2023-raw_collocated-{y}.pq"
CONFIG_PATH = ROOT / "scripts/config/example_training_config.yaml"


LOGGING_LEVEL = "DEBUG"
#CV_VERBOSITY = 3
N_CPUS = -1  # -1 = all available CPUs

DF_INDEX_COLUMNS = (
    "expocode",
    "time",
    "lat",
    "lon",
)

COMPULSORY_COLUMNS = {
    "talk",
    "salinity",
} | set(DF_INDEX_COLUMNS)

ALL_FEATURES = (
    "salinity",
    "temperature",
    "bottomdepth",
    "mld_dens_soda",
    "ssh_adt",
    "ssh_sla",
    "chl_globcolour"
) #FIXME: should this be a set or a tuple or a list?


LIN_BASELINE_PARAM = {
    'fit_intercept': True,
    'copy_X': True,
    'tol': 1e-06,
    'n_jobs':None, 
    'positive': False
}

SALINITY_BIN_EDGES = (
    0,
    32,
    34,
    36,
    np.inf,
)

DROP_AT_IMPORT = [
    "chl",
    "chl_filled_flag",
    "chl_flag",
    "chl_sigma",
    "chl_sigma_bias",
    "chl_sigma_regrid",
    "chl_sigma_uncert",
    "fco2atm_noaa",
    "ice",
    "pres_std",
    "press",
    "ssh_sigma",
    "ssh_sigma_regrid",
    "ssh_sigma_uncert",
    "sss_flag",
    "sss_old",
    "sss_old_flag",
    "sss_sigma",
    "sss_sigma_uncert",
    "sst_flag",
    "sst_sigma",
    "sst_sigma_regrid",
    "sst_sigma_uncert",
    "wind_std",
    "windspeed_moment1",
    "windspeed_moment2",
    "xco2atm_mauna_loa",
    "xco2mbl_noaa",
]

logger.remove()
logger.add(sys.stderr, level=LOGGING_LEVEL)

# %%

@dataclass
class TrainingConfig:

    save_folder: str
    training_mode: Literal['GridSearch','SingleRun'] # two training pipelines
    catboost_param_grid: dict 
      
    num_cv_folds: int = 5
    
    run_param_single: dict[str, object] | None = None #could be aggregated in one single run_parameter
    run_param_gridsearch: dict[str, object]  | None = None
        
    xname_features: tuple[str, ...] = ALL_FEATURES
    yname_target: str = 'talk_normalized'
    
    salinity_bins: tuple[float, ...] = SALINITY_BIN_EDGES
    salinity_name: str = "salinity"
    salinity_norm_value: float = 34.5
    
    baseline_regression: Literal["LinearRegression","None"] = 'LinearRegression' # do we want possibility for ridge, lasso etc or only numpy?
    baseline_features: tuple[str, ...] = ALL_FEATURES
    #baseline_param: dict[str, object] = field(default= LIN_BASELINE_PARAM)
    
    
    
# 
    
def make_cb_pool(config, df) -> cb.Pool:
    
    cb_pool = cb.Pool(data = df[list(config.xname_features)],
                         label= df[config.yname_target])

    
    return cb_pool

def load_config() -> TrainingConfig:
    global SAVE_PATH

    config_path = CONFIG_PATH
    with open(config_path, "r") as f:
        config_dict = yaml.safe_load(f)

    config = TrainingConfig(**config_dict)


    SAVE_PATH = ROOT / config.save_folder
    SAVE_PATH.mkdir(parents=True, exist_ok=True)

    NFOLDS = config.num_cv_folds #maybe dont define NFOLDS before anything else
    
    
    if not config.training_mode:
        raise ValueError("No training mode provided in configuration file")
    
    #TODO: other verification ifs
    
    logger.debug(f"Training configuration: {config}")
    logger.success(f"Loaded config and created output folder at: {SAVE_PATH}")
    
    return config

#%%
def load_data(compulsory_columns: set[str] = COMPULSORY_COLUMNS) -> pd.DataFrame:
    
    """
    Loads the training data from a parquet file

    Contains all the columns and rows of the training data
    No engineering or preprocessing is done here, just loading
    the data into a pandas dataframe

    Parameters
    ----------
    fname_data_parquet : str | Path
        The path to the parquet file containing the training data
    Returns
    -------
    pd.DataFrame
        The training data as a pandas dataframe
    """
    
    data_path = str(DATA_PATH)

    logger.info(f"Loading data from {data_path.format(y='YYYY')} for years 1982-2021")
    data = pd.concat([pd.read_parquet(data_path.format(y=y)) for y in range(1982, 2022)],join="outer")

    # Check that the compulsory columns are present in the data
    columns = data.columns.intersection(compulsory_columns)

    if len(columns) < len(compulsory_columns):
        missing_cols = compulsory_columns - set(columns)
        raise ValueError(f"Missing columns in the data: {missing_cols}")
    
    logger.success("Loaded data")

    logger.debug(f"data.head() = \n{data.head().head(50)}")
    
    return data



def preprocess_data(df: pd.DataFrame, config: TrainingConfig) -> pd.DataFrame:
    
    """
    Select columns, engineer features, bin salinity

    Parameters
    ----------
    df : pd.DataFrame
        The raw training df as a pandas dataframe
    config : ModelSelectionConfig
        The configuration for the model selection process, containing any parameters needed for preprocessing

    Returns
    -------
    pd.DataFrame
        The preprocessed training df
    """

    # salinity binning
    salinity_bins = config.salinity_bins
    salinity = df[config.salinity_name]
    df["salinity_bin"] = salinity_binning(salinity, bins=salinity_bins)
    
    # alkalinity normalization
    salt_norm_value = config.salinity_norm_value
    df["talk_normalized"] = normalize_alkalinity(df["talk"], salinity, salt_norm_value)
    
    # filter outliers
    df = filter_outliers(df, column="salinity", lower_abs=20, upper_abs=40)

    # set quadruple index 
    index_columns = DF_INDEX_COLUMNS
    index_columns = list(DF_INDEX_COLUMNS + ("salinity_bin",))
    df = df.set_index(index_columns)

    # select columns and drop rows with missing values in the selected columns
    keep_cols = set(list(config.xname_features) + [config.yname_target]).union(COMPULSORY_COLUMNS)
    valid_columns = list(keep_cols - set(index_columns))
    df = df[valid_columns].dropna()


    logger.debug(f"Preprocessed data: {df.describe}")
    logger.success(f"Peprocessed data with kept column {valid_columns} and index: {index_columns}")

    return df

#%%
def filter_outliers(df: pd.DataFrame, column: str, lower_abs: float, upper_abs: float) -> pd.DataFrame:
    if lower_abs is not None:
        df = df[df[column] >= lower_abs]
    if upper_abs is not None:
        df = df[df[column] <= upper_abs]

    return df

#%%
def salinity_binning(
    salinity: pd.Series, bins: tuple[float, ...], bin_labels: None | list[str | float] = None
) -> pd.Series:
    n_bins = len(bins)
    bin_label = bin_labels or range(1, n_bins)
    return pd.cut(salinity, bins=bins, labels=bin_label)


#%%
def normalize_alkalinity(alkalinity: pd.Series, salinity: pd.Series, norm_value: float) -> pd.Series:
    return norm_value * alkalinity / salinity


#%%
def get_splits_by_expocode_salinity_bin_based(data: pd.DataFrame, random_state: int = 42, n_folds: int = 5):
    
    index = data.index.to_frame()

    grouper = index["expocode"]
    stratifier = index["salinity_bin"]

    #TODO:verify how the shuffle affects stratification and grouping
    # PROBLEM: default random_state is 42, so shuffle will always be True.
    
    shuffle = False if random_state is None else True  # if random state provided, then True
    #splitter = StratifiedGroupKFold(n_splits=n_folds, shuffle=shuffle, random_state=random_state)
    
    splitter = StratifiedGroupKFold(n_splits=n_folds)
    logger.debug(f"Splitter with {n_folds} folds: {splitter}, type {type(splitter)}")

    splits = splitter.split(data, y=stratifier, groups=grouper)

    logger.debug(f"Splits with {n_folds} folds: {splits}, type {type(splits)}")
    
    # return a list so that we can pickle the CV splitter later
    
    return splits

#%%
def get_train_test_split_by_expocode_salinity_bin_based(
    data: pd.DataFrame, random_state: int = 42) -> tuple[pd.DataFrame, pd.DataFrame]:

    # Generate a 5-fold StratifiedGroupKFold, yielding a 20% validation and 80% train distribution in each of the 5 folds
    train_test_splitter = get_splits_by_expocode_salinity_bin_based(data, random_state=random_state)
    
    train_test_splitter_list = list(train_test_splitter)
    
    # only take the first fold instance to generate our train-test split according to its train-validation (80:20) distribution 
    idx_train, idx_test = train_test_splitter_list[0]
    
    # generate train and test dfs according to the indices
    train = data.iloc[idx_train]
    test = data.iloc[idx_test]

    return train, test
    

#%% 
def train_baseline(X : pd.DataFrame, y : pd.Series, baseline_param: dict
                   ) ->  LinearRegression:

    lin_regression = LinearRegression(**baseline_param)
    lin_regression = lin_regression.fit(X, y)
    
    logger.debug(f"Regressor coefs: {lin_regression.coef_}")
    logger.success("Trained linear regression baseline")

    return lin_regression

def predict_baseline(config, train_df, test_df) -> tuple[LinearRegression, np.ndarray, np.ndarray]:
    
    train_x_base = train_df[list(config.baseline_features)]
    test_x_base = test_df[list(config.baseline_features)]
        
    baseline_regressor = train_baseline(
            X= train_x_base, 
            y= train_df[config.yname_target], 
            baseline_param=LIN_BASELINE_PARAM)          
        
    x_train_pred = baseline_regressor.predict(train_x_base)
    x_test_pred = baseline_regressor.predict(test_x_base)
    
    logger.debug(f"Linear prediction array (shape:{x_train_pred.size}): {x_train_pred}")
    logger.debug(f"Linear prediction array (shape:{x_test_pred.size}): {x_test_pred}")
    
    if x_train_pred.size != train_x_base.shape[0]:
        raise IndexError(f"Training sample size {train_x_base.shape[0]} not equal to baseline predictions size {x_train_pred.size}")
    
    if x_test_pred.size != test_x_base.shape[0]:
        raise IndexError(f"Training sample size {test_x_base.shape[0]} not equal to baseline predictions size {x_test_pred.size}")
    
    return baseline_regressor, x_train_pred, x_test_pred
    
def gridsearch_train_catboost(config : TrainingConfig, 
                              train_pool : cb.Pool, 
                              cv_splitter : StratifiedGroupKFold | Iterator[tuple[np.ndarray, np.ndarray]]
                              ) -> cb.CatBoostRegressor: #FIXME: the generator type
    

    cb_regressor = cb.CatBoostRegressor()
    
    cb_regressor.grid_search(
        param_grid = config.catboost_param_grid,
        X = train_pool,
        cv = cv_splitter,
        refit = True
    )
    #maybe do a gridsearch_run_param
    
    logger.info(f"CV_folds_scores: {cb_regressor.best_score_}")
    logger.success(f"Trained regressor")
    
    return cb_regressor


def predict_on_test(cb_model, test_pool):
    
    from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score, median_absolute_error

    y_pred = cb_model.predict(test_pool, verbose = True)
    test_y = test_pool.get_label()

    # dataframe with scores on metrics from CV scoring metrics
    
    scores = pd.DataFrame({
        "root_mean_squared_error": [root_mean_squared_error(test_y, y_pred)],
        "median_absolute_error": [median_absolute_error(test_y, y_pred)],
        "mean_absolute_error": [mean_absolute_error(test_y, y_pred)],
        "r2": [r2_score(test_y, y_pred)],
    })
    
    #publish_test_scores(scores, cv_model.estimator.__class__.__name__)
    
    logger.info(f"Test score for model: {scores.T.to_markdown()}")
    #logger.info(f"Test score per sample for model {cv_model.estimator.__class__.__name__}: {score_sample}")
    
    return y_pred, scores

# def publish_test_scores(scores, model_name: str):
#     fig, ax = plt.subplots(figsize=(10, 4))
#     ax.axis('tight')
#     ax.axis('off')
#     table = ax.table(cellText=scores.values, colLabels=scores.columns, rowLabels=scores.index, cellLoc='center', loc='center')
#     table.auto_set_font_size(False)
#     table.set_fontsize(10)
#     table.scale(1.2, 1.5)
    
#     plt.title("Test Scores of Best Estimator on Test Set", fontsize=14)
#     plt.savefig(SAVE_PATH / f"{model_name}_test_scores.png", bbox_inches='tight')
    
#     logger.success(f"Saved test scores to {SAVE_PATH} / {model_name}_test_scores.png")
    
# def single_train_catboost(config : TrainingConfig, 
#                           X: pd.DataFrame, y : pd.DataFrame, 
# ):
    
#     # maybe a safety check on the param_grid: should only have single values for each key
    
#     cb_regressor = cb.CatBoostRegressor([*config.param_grid,*config.run_param_single])
    
#     cb_regressor.fit(X, y)
    
    
#def failsafe_checks():

    estimator_names = set(ESTIMATOR_NAMES.__args__)
    estimator_keys = set(ESTIMATORS.keys())

    assert estimator_names == estimator_keys, (
        f"ESTIMATOR_NAMES and ESTIMATORS keys must match, but got {estimator_names} and {estimator_keys}"
    )


@lru_cache(maxsize=1)
def load_inference_data() -> xr.Dataset:
    
    
    url = "https://data.up.ethz.ch/shared/OceanSODA-ETHZv2/inference_for_gregor2024/data_8daily_25km_v01.zarr/"
    ds_raw = xr.open_zarr(url, consolidated=True, group="2004", drop_variables=DROP_AT_IMPORT)

    url = "https://www.ngdc.noaa.gov/thredds/dodsC/global/ETOPO2022/60s/60s_bed_elev_netcdf/ETOPO_2022_v1_60s_N90W180_bed.nc"

    #bath = xr.open_dataset(url, engine="netcdf4")["z"]
    #bath = xr.open_dataset("https://www.ngdc.noaa.gov/thredds/fileServer/global/ETOPO2022/60s/60s_bed_elev_netcdf/ETOPO_2022_v1_60s_N90W180_bed.nc", chunks={})["z"]
    #bath = bath.coarsen(lat=15, lon=15).mean().interp_like(ds_raw).rename('bottomdepth').compute()

    #bath.to_netcdf("bath_coarse.nc")
    
    ds = ds_raw
    #ds = xr.merge([ds_raw[["sss", "sst", "ssh", "ssh_anom", "mld"]], bath])
    
    ds = ds.rename(
        sss="salinity", sst="temperature", ssh="ssh_adt", ssh_anom="ssh_sla", mld="mld_dens_soda"
    )
    return ds


def preprocess_inference_data(inference_dataset : xr.Dataset, train_columns: list[str], t = 0) -> pd.DataFrame:

    #if "df" not in locals() or t != t_prev or (df.columns.difference(train_x.columns)).any():

    inference_df = inference_dataset.isel(time=[t]).to_dataframe()
    coords = inference_df.index.to_frame()
    inference_df["lat"] = coords["lat"]
    inference_df["lon_sin"] = np.sin(np.radians(coords["lon"]))
    inference_df["lon_cos"] = np.cos(np.radians(coords["lon"]))
    day_of_year = coords["time"].dt.dayofyear
    inference_df["sin_doy"] = np.sin(2 * np.pi * day_of_year / 365.25)
    inference_df["cos_doy"] = np.cos(2 * np.pi * day_of_year / 365.25)

    #t_prev = t

    inference_df = inference_df.rename(columns = {'chl_filled':'chl_globcolour'})[train_columns]
    dfnn = inference_df.dropna()
    
    
    logger.debug(f"Inference dataframe: {inference_df.to_markdown()}")
    logger.success("Preprocessed inference data")
    return dfnn


def predict_cb_inference(inference_df: pd.DataFrame, model: cb.CatBoostRegressor, baseline_pred = None):
    
    #loop on t
    if baseline_pred != None:
        inference_pool = cb.Pool(data = inference_df, baseline= baseline_pred)
    else:
        inference_pool = cb.Pool(data = inference_df)
        
    predictions = model.predict(inference_pool)
    
    logger.success("Predictions on inference data")
    
    
    inference_predictions_ds = pd.DataFrame({'ta_pred': predictions}, index = inference_df.index).to_xarray()    
    
    return inference_predictions_ds


def plot_inference_map(dataset : xr.Dataset, var : str = 'ta_pred', t : int = 0):
    
    
    import matplotlib.pyplot as plt
    
    props_quickmap = dict(aspect=2.3, size=5, robust=True, cbar_kwargs=dict(pad=0.02))

    da = dataset[var].isel(time=t)

    da.plot(
        x="lon",
        y="lat",
        **props_quickmap
    )
    plt.show();


In [3]:
# load config
config = load_config()

# load and preprocess data
data = load_data()
data = preprocess_data(data, config)

inference_dataset = load_inference_data()
inference_df = preprocess_inference_data(inference_dataset, list(config.xname_features))

2026-04-13 11:36:40.656 | DEBUG    | __main__:load_config:176 - Training configuration: TrainingConfig(save_folder='outputs/training/example_training', training_mode='GridSearch', catboost_param_grid={'iterations': [5, 10, 20], 'depth': [3, 6]}, num_cv_folds=5, run_param_single=None, run_param_gridsearch=None, xname_features=['salinity', 'temperature', 'mld_dens_soda', 'ssh_adt', 'ssh_sla', 'chl_globcolour'], yname_target='talk_normalized', salinity_bins=(0, 32, 34, 36, inf), salinity_name='salinity', salinity_norm_value=34.5, baseline_regression='None', baseline_features=['temperature', 'salinity'])
2026-04-13 11:36:40.658 | SUCCESS  | __main__:load_config:177 - Loaded config and created output folder at: /home/edupuis/highres_TA/outputs/training/example_training
2026-04-13 11:36:40.660 | INFO     | __main__:load_data:203 - Loading data from /home/edupuis/highres_TA/data/training/GLODAPv2023-raw_collocated-YYYY.pq for years 1982-2021
2026-04-13 11:36:41.672 | SUCCESS  | __main__:load_

OSError: [Errno -90] NetCDF: file not found: 'https://www.ngdc.noaa.gov/thredds/fileServer/global/ETOPO2022/60s/60s_bed_elev_netcdf/ETOPO_2022_v1_60s_N90W180_bed.nc'

In [ ]:
# train-test split and CV splitter generation
train_df, test_df = get_train_test_split_by_expocode_salinity_bin_based(data)
cv_splitter = get_splits_by_expocode_salinity_bin_based(train_df) 

# prepare training pools
train_pool= make_cb_pool(config, train_df)
test_pool = make_cb_pool(config, test_df)

if config.baseline_regression == 'LinearRegression': #maybe a possibility of different linear regressions
    baseline_regressor, train_baseline, test_baseline = predict_baseline(config=config, train_df= train_df, test_df= test_df)
    train_pool.set_baseline(baseline=train_baseline)
    test_pool.set_baseline(baseline=test_baseline)
    
catboost_model = gridsearch_train_catboost(config=config, train_pool = train_pool, cv_splitter = cv_splitter)

y_test_pred, test_scores = predict_on_test(catboost_model, test_pool)

baseline_inference = None
if config.baseline_regression == 'LinearRegression':
    baseline_inference = baseline_regressor.predict(inference_df)
    
inference_predictions_ds = predict_cb_inference(inference_df, catboost_model, baseline_pred=baseline_inference)

plot_inference_map(inference_predictions_ds)

# if config.training_mode == 'SingleRun':
#     trained_cb = single_train_catboost(config, X= train_x, y = train_y)
# elif config.training_mode == 'GridSearch':
#     trained_cb = gridsearch_train_catboost(config, X= train_x, y = train_y, cv_splitter=cv_splitter)

In [ ]:
# inference_predictions_df = pd.DataFrame({'ta_pred': inference_predictions}, index = inference_df.index, )
# inference_predictions_ds = inference_predictions_df.to_xarray()
#inference_predictions_ds = pd.DataFrame({'ta_pred': inference_predictions}, index = inference_df.index).to_xarray() 
props_quickmap = dict(aspect=2.3, size=5, robust=True, cbar_kwargs=dict(pad=0.02))
img = inference_dataset.salinity[0].plot.imshow(**props_quickmap)
plt.show();

In [ ]:
plot_inference_map(inference_dataset, 'salinity', t=0)

# Local data download

In [ ]:
import requests


DATA_PATH = ROOT / "data/training/"
url = "https://data.up.ethz.ch/shared/OceanSODA-ETHZv2/total_alkalinity/GLODAPv2023/GLODAPv2023-raw_collocated-{y}.pq"

for y in range(1982, 2022):
    # Download the files in the data folder
    (DATA_PATH / f"GLODAPv2023-raw_collocated-{y}.pq").write_bytes(requests.get(url.format(y=y)).content)


In [ ]:
glodap_path2004 = DATA_PATH / f"GLODAPv2023-raw_collocated-1982.pq"
test_rawdata2004 = pd.read_parquet(glodap_path2004)
test_rawdata2004.columns

# Imports

In [ ]:
from sklearn import model_selection, metrics
from glob import glob
import matplotlib.pyplot as plt

import numpy as np
import pandas as pd
import catboost as cb

import sys
from pathlib import Path

current_path = Path.cwd()

parent_path = current_path.parent

sys.path.append(f"{parent_path}/src/highres_ta")

from highres_ta import preprocessing_tools as ppt
from highres_ta import modelling_tools as mot

plt.style.use('default')

In [ ]:
# df = ppt.online_GLODAP_import

df = ppt.online_GLODAP_import(parent_path)

salinity_bin_edges = [0, 32, 34, 36, np.inf]
salinity_bin_labels = ['<32', '32-34', '34-36', '>36']
df['salinity_bin'] = pd.cut(df['salinity_gp'], bins=salinity_bin_edges, labels=salinity_bin_labels)
df['talk_gp_normalized'] = 35*df['talk_gp']/df['salinity_gp']

df = df.dropna(subset=['salinity_bin'])

df.salinity_gp.describe()

# Train-test split

In [ ]:
train_split_df, heldout_test_df = ppt.traintest_salinity_expocode_split(df)

y_train = train_split_df['talk_gp']
x_train = ppt.keep_predictors(train_split_df)

y_test = heldout_test_df['talk_gp']
x_test = ppt.keep_predictors(heldout_test_df)

In [ ]:
cv_folds_dict = ppt.cv_salinity_expocode_split(train_split_df, n_splits=5, plot_cv_splits=True)

In [ ]:
normalized_cv_folds_dict = ppt.cv_salinity_expocode_split(train_split_df, n_splits=5, normalized_y=True, plot_cv_splits=True)

In [ ]:
# # save train-test split to files

# x_train.to_parquet("X_train.parquet")
# x_test.to_parquet("X_test.parquet")
# y_train.to_parquet("y_train.parquet")
# y_test.to_parquet("y_test.parquet")

# Models intercomparison


In [ ]:
import catboost as cb
from sklearn.ensemble import RandomForestRegressor

In [ ]:
train_dir = parent_path / f"trained_models"
rf_dir = train_dir/f"RandomForest"
gp_dir = train_dir/f"GaussianProcesses"
cb_dir = train_dir/f"Catboost"

In [ ]:
test_experiment = mot.Experiment(cv_folds_dict, train_dir, 'not_normalized')
test_experiment.add_a_new_model(regressor = RandomForestRegressor(n_estimators=5, random_state=42), model_name = 'SimpleRandomForest')
test_experiment.add_a_new_model(regressor = cb.CatBoostRegressor(iterations=5, random_seed=42, loss_function='RMSE',verbose=0), model_name = 'SimpleCatBoost_RMSE')
test_experiment.add_a_new_model(regressor = cb.CatBoostRegressor(iterations=5, random_seed=42, loss_function='RMSEWithUncertainty',verbose=0), model_name = 'SimpleCatBoost_RMSEUncert')
test_experiment.run_full_cv_assessment()


In [ ]:
normalized_test_experiment = mot.Experiment(normalized_cv_folds_dict, train_dir, 'normalized')
normalized_test_experiment.add_a_new_model(regressor = RandomForestRegressor(n_estimators=5, random_state=42), model_name = 'SimpleRandomForest')
normalized_test_experiment.add_a_new_model(regressor = cb.CatBoostRegressor(iterations=5, random_seed=42, loss_function='RMSE',verbose=0), model_name = 'SimpleCatBoost_RMSE')
normalized_test_experiment.add_a_new_model(regressor = cb.CatBoostRegressor(iterations=5, random_seed=42, loss_function='RMSEWithUncertainty',verbose=0), model_name = 'SimpleCatBoost_RMSEUncert')
normalized_test_experiment.run_full_cv_assessment()

In [ ]:
# model_dir.mkdir(exist_ok=True)
# rf_dir.mkdir(exist_ok=True)
# cb_dir.mkdir(exist_ok=True)
# gp_dir.mkdir(exist_ok=True)

huber score

In [ ]:
# from sklearn.model_selection import learning_curve


# def plot_learning_curve(model, X, y):

#     train_sizes, train_scores, val_scores = learning_curve(
#         model,
#         X,
#         y,
#         cv=5,
#         scoring="neg_root_mean_squared_error"
#     )

#     train_mean = -train_scores.mean(axis=1)
#     val_mean = -val_scores.mean(axis=1)

#     plt.figure(figsize=(7,5))

#     plt.plot(train_sizes, train_mean, label="Train")
#     plt.plot(train_sizes, val_mean, label="Validation")

#     plt.xlabel("Training Set Size")
#     plt.ylabel("RMSE")

#     plt.title("Learning Curve")
#     plt.legend()

#     plt.show()
    
# plot_learning_curve(model_RandomForest, x_train, y_train)

In [ ]:
# def plot_feature_importance(feature_names, importance):

#     import seaborn as sns
    
#     df = pd.DataFrame({
#         "feature": feature_names,
#         "importance": importance
#     })

#     df = df.sort_values("importance", ascending=False)

#     plt.figure(figsize=(7,6))


#     sns.barplot(
#         data=df.head(20),
#         x="importance",
#         y="feature"
#     )

#     plt.title("Feature Importance (Top 20)")

#     plt.show()
    


In [ ]:
# from sklearn.ensemble import RandomForestRegressor
# import modelling_tools as mot
# import shutil

# folder_to_delete = parent_path / f"trained_models/SimpleRandomForest_crossval"
# shutil.rmtree(folder_to_delete)

# test_config = {
#     'regressor' : RandomForestRegressor(n_estimators=5, random_state=42),
#     'model_name': 'SimpleRandomForest_crossval',
#     'parent_dir': parent_path/f"trained_models"
#     }
    

# test_wrapper = mot.ModelWrapper(**test_config)


# # x_train = ppt.keep_predictors(train_split_df)
# # y_train = train_split_df['talk_gp']
# # x_eval = ppt.keep_predictors(heldout_test_df)
# # y_eval = heldout_test_df['talk_gp']

# test_wrapper.cross_validation_run(cv_folds_dict, export_fold_metadata=False)

# Model Intercomparison v2

Catboost

In [ ]:
import catboost as cb

In [ ]:
model = cb.CatBoostRegressor(iterations=2_000, loss_function="RMSEWithUncertainty")
model = model.fit(x_train, y_train, plot=True)

yhat_train, yhat_train_std = model.predict(x_train).T
yhat_test, yhat_test_std = model.predict(x_test).T

In [ ]:
df_metrics = pd.DataFrame({
    'r2_score': {
        'train': metrics.r2_score(y_train, yhat_train), 
        'test': metrics.r2_score(y_test, yhat_test)
    },
    'rmse': {
        'train': metrics.root_mean_squared_error(y_train, yhat_train), 
        'test': metrics.root_mean_squared_error(y_test, yhat_test)
    },
    'mae': {
        'train': metrics.mean_absolute_error(y_train, yhat_train), 
        'test': metrics.mean_absolute_error(y_test, yhat_test)
    },
    'bias': {
        'train': (yhat_train - y_train).mean(), 
        'test': (yhat_test - y_test).mean()
    }
})

df_metrics.T.round(2)

In [ ]:
names: list[str] = model.feature_names_  # type: ignore
values = model.feature_importances_

pd.Series({k: float(v) for k, v in zip(names, values, strict=True)}).sort_values(ascending=False)

RandomForest


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn import model_selection
from sklearn import metrics

In [ ]:
PARAMETERS = {
    "n_estimators": [100],
    "max_depth": [10, None],
    "min_samples_split": [5, 10],
}


parameters = PARAMETERS
model = RandomForestRegressor(n_jobs=-1)

grid_search = model_selection.GridSearchCV(
    estimator=model, param_grid=parameters, cv=5, scoring="neg_mean_squared_error", verbose=2
)

grid_search.fit(
    x_train,
    y_train,
)

In [ ]:
yhat_train = grid_search.best_estimator_.predict(x_train)
yhat_test = grid_search.best_estimator_.predict(x_test)

In [ ]:
df_metrics = pd.DataFrame(
    {
        "r2_score": {
            "train": metrics.r2_score(y_train, yhat_train),
            "test": metrics.r2_score(y_test, yhat_test),
        },
        "rmse": {
            "train": metrics.root_mean_squared_error(y_train, yhat_train),
            "test": metrics.root_mean_squared_error(y_test, yhat_test),
        },
        "mae": {
            "train": metrics.mean_absolute_error(y_train, yhat_train),
            "test": metrics.mean_absolute_error(y_test, yhat_test),
        },
        "bias": {"train": (yhat_train - y_train).mean(), "test": (yhat_test - y_test).mean()},
    }
)

df_metrics.T.round(2)
